# 02 · Dinámica de ligas: ascensos y descensos

Tras la clasificatoria, las estrategias se reparten en ligas (Liga 1 = la mejor). Cada temporada los 2 primeros de cada liga suben y los 2 últimos bajan. Aquí vemos **qué estrategias se asientan arriba** cuando solo compiten contra rivales de su nivel.

In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from split_steal.runner import load_run
from split_steal import plots

# Run canónico versionado en el repo. Para regenerarlo:
#   split-steal tournament --name canonical --out-dir results/canonical
RUN_DIR = Path("../results/canonical/canonical")
result = load_run(RUN_DIR)
result.config

TournamentConfig(roster='full', n_leagues=5, n_per_league=6, rounds=100, n_seasons=20, n_ascend_descend=2, seed=42, qualifying=True)

## Liga de cada estrategia por temporada

In [2]:
plots.plot_leagues_by_season(result);

## Liga media y número de cambios de liga

In [3]:
st = result.standings
seasons = st[st["season"] > 0].sort_values(["name", "season"])
moves = seasons.groupby("name")["league"].apply(lambda s: int((s.diff().abs() > 0).sum()))
table = (seasons.groupby("name")["league"].agg(liga_media="mean", mejor="min", peor="max")
         .assign(cambios=moves).sort_values("liga_media"))
table.round(2)

,liga_media,mejor,peor,cambios
name,,,,
CopycatDelayed,1.00,1,1,0
ForgiveAfter3,1.00,1,1,0
ForgivingTitForTat,1.00,1,1,0
ConditionalRandom,1.05,1,2,1
GrimTrigger,1.45,1,2,18
RevengeOneRound,1.60,1,2,16
MimicMajority,1.65,1,3,18
TitForTat,1.90,1,3,18
Mirror2Rounds,2.00,1,3,3


## Clasificatoria vs. ligas

¿Quien gana contra todos es también quien se queda en Liga 1? Comparamos el puesto en la clasificatoria con la liga media.

In [4]:
q_pos = st[st["season"] == 0].set_index("name")["position"]
comparison = table[["liga_media"]].join(q_pos.rename("puesto_clasificatoria"))
comparison.sort_values("puesto_clasificatoria")

,liga_media,puesto_clasificatoria
name,,
GrimTrigger,1.45,1
SplitUntilBetrayedTwice,2.45,2
ForgiveAfter3,1.00,3
TwoTitsForTat,2.50,4
ForgivingTitForTat,1.00,5
CopycatDelayed,1.00,6
ConditionalRandom,1.05,7
ProbTitForTat,2.30,8
RevengeOneRound,1.60,9


## Empates en la Liga 1

Cuando en una liga todas cooperan siempre, empatan a puntos y el desempate es **alfabético** (decisión del simulador para que sea determinista). Eso decide quién desciende de la Liga 1:

In [5]:
last = st["season"].max()
st[(st["season"] == last) & (st["league"] == 1)]

,season,league,name,points,position
600,20,1,ConditionalRandom,1500,1
601,20,1,CopycatDelayed,1500,2
602,20,1,ForgiveAfter3,1500,3
603,20,1,ForgivingTitForTat,1500,4
604,20,1,GrimTrigger,1500,5
605,20,1,MimicMajority,1500,6
